In [ ]:
# 02_study1_polish.ipynb
# Study 1 (Polish bankruptcy): does a parsimonious 2-indicator interim signal add
# incremental default-prediction power beyond a base grade built from the other features?
# Tests H1/H2 (incremental AUC, DeLong, LR), H3 (asymmetry), H4 (band moderation),
# and the pseudo-time gradient across forecast horizons.
# Base grade uses 62 features EXCLUDING the 2 proxies, so the proxy signal is genuinely new.
# All code in a single cell; grayscale figures saved as png + pdf at dpi 600.

import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.io import arff
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
import statsmodels.api as sm
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

DATA = "../data"; FIG = "../results/figures"; TAB = "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="white", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8, "axes.grid": False})
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}/{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight"); plt.close(fig)

# ---- load and pool the five horizons ----
def load_year(n):
    data, _ = arff.loadarff(f"{DATA}/polish/{n}year.arff"); df = pd.DataFrame(data)
    cls = [c for c in df.columns if c.lower() == "class"][0]
    df[cls] = df[cls].apply(lambda b: int(b.decode()) if isinstance(b, bytes) else int(b))
    feats = [c for c in df.columns if c != cls]
    df.columns = [f"Attr{i+1}" for i in range(len(feats))] + ["target"]; df["horizon"] = n
    return df
pooled = pd.concat([load_year(n) for n in range(1, 6)], ignore_index=True)
feats = [c for c in pooled.columns if c.startswith("Attr")]
proxies = ["Attr1", "Attr2"]                       # ROA (profitability), leverage
base_feats = [c for c in feats if c not in proxies]  # 62 features
y = pooled["target"].astype(int).values
Xall = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(pooled[feats]), columns=feats)
Hd = pd.get_dummies(pooled["horizon"], prefix="h", drop_first=True).astype(float).reset_index(drop=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# ---- DeLong test for two correlated ROC AUCs ----
def delong(y_true, p1, p2):
    y_true = np.asarray(y_true); pos = y_true == 1; neg = ~pos; m = pos.sum(); n = neg.sum()
    def st(p):
        xp, xn = p[pos], p[neg]
        V10 = np.array([(np.sum(xn < xi) + 0.5 * np.sum(xn == xi)) / n for xi in xp])
        V01 = np.array([(np.sum(xp > xj) + 0.5 * np.sum(xp == xj)) / m for xj in xn])
        return V10.mean(), V10, V01
    a1, v10a, v01a = st(p1); a2, v10b, v01b = st(p2)
    S10 = np.cov(np.vstack([v10a, v10b])); S01 = np.cov(np.vstack([v01a, v01b]))
    var = S10 / m + S01 / n; v = var[0, 0] + var[1, 1] - 2 * var[0, 1]
    z = (a1 - a2) / np.sqrt(v) if v > 0 else 0.0
    return float(a1), float(a2), float(z), float(2 * (1 - stats.norm.cdf(abs(z))))

# ---- base grade G (from 62 features, OOF) and 2-proxy signal S (OOF) ----
base_pd = cross_val_predict(LogisticRegression(max_iter=3000, class_weight="balanced"),
                            StandardScaler().fit_transform(Xall[base_feats]), y, cv=cv, method="predict_proba")[:, 1]
G = pd.qcut(pd.Series(base_pd), 7, labels=False, duplicates="drop").values
S = cross_val_predict(LogisticRegression(max_iter=3000, class_weight="balanced"),
                      StandardScaler().fit_transform(Xall[proxies]), y, cv=cv, method="predict_proba")[:, 1]
D = pd.DataFrame({"y": y, "G": G.astype(float), "S": S})

# ---- M0 (base grade) vs M1 (+ 2-proxy): OOF probabilities ----
M0X = pd.concat([D[["G"]], Hd], axis=1).values
M1X = pd.concat([D[["G", "S"]], Hd], axis=1).values
p0 = cross_val_predict(LogisticRegression(max_iter=3000), M0X, y, cv=cv, method="predict_proba")[:, 1]
p1 = cross_val_predict(LogisticRegression(max_iter=3000), M1X, y, cv=cv, method="predict_proba")[:, 1]
auc0, auc1 = roc_auc_score(y, p0), roc_auc_score(y, p1)
a1, a0, z, pv = delong(y, p1, p0)
m0 = sm.Logit(D["y"], sm.add_constant(pd.concat([D[["G"]], Hd], axis=1).astype(float))).fit(disp=0)
m1 = sm.Logit(D["y"], sm.add_constant(pd.concat([D[["G", "S"]], Hd], axis=1).astype(float))).fit(disp=0)
LR = 2 * (m1.llf - m0.llf); lr_p = stats.chi2.sf(LR, 1)
h1h2 = pd.DataFrame([
    {"model": "M0 (base grade)", "AUC": round(auc0, 4), "AR_Gini": round(2*auc0-1, 4), "Brier": round(brier_score_loss(y, p0), 5)},
    {"model": "M1 (+2-proxy)",  "AUC": round(auc1, 4), "AR_Gini": round(2*auc1-1, 4), "Brier": round(brier_score_loss(y, p1), 5)},
])
h1h2.to_csv(f"{TAB}/study1_h1h2.csv", index=False)
print("H1/H2:\n", h1h2.to_string(index=False))
print(f"DeLong z = {z:.2f}, p = {pv:.2e} | LR = {LR:.1f}, p = {lr_p:.2e} | coef_S = {m1.params['S']:.3f} (z = {m1.tvalues['S']:.2f})")

# ---- H3 asymmetry: deviation of S from grade-peer mean ----
D["dev"] = D["S"] - D.groupby("G")["S"].transform("mean")
D["det"] = D["dev"].clip(lower=0); D["imp"] = D["dev"].clip(upper=0)
m3 = sm.Logit(D["y"], sm.add_constant(pd.concat([D[["G", "det", "imp"]], Hd], axis=1).astype(float))).fit(disp=0)
# ---- H4 band moderation ----
D["mid"] = np.isin(G, [2, 3, 4]).astype(float); D["S_mid"] = D["S"] * D["mid"]
m4 = sm.Logit(D["y"], sm.add_constant(pd.concat([D[["G", "S", "mid", "S_mid"]], Hd], axis=1).astype(float))).fit(disp=0)
hyp = pd.DataFrame([
    {"test": "H3 deterioration (coef, z)", "value": f"{m3.params['det']:.3f}, {m3.tvalues['det']:.2f}"},
    {"test": "H3 improvement (coef, z)",   "value": f"{m3.params['imp']:.3f}, {m3.tvalues['imp']:.2f}"},
    {"test": "H4 S x mid-band (coef, z)",  "value": f"{m4.params['S_mid']:.3f}, {m4.tvalues['S_mid']:.2f}"},
])
hyp.to_csv(f"{TAB}/study1_h3h4.csv", index=False)
print("\nH3/H4:\n", hyp.to_string(index=False))

# ---- pseudo-time gradient: OOF AUC by horizon ----
grad = []
for h in range(1, 6):
    msk = pooled["horizon"].values == h
    grad.append({"horizon": h, "AUC_M0": round(roc_auc_score(y[msk], p0[msk]), 4),
                 "AUC_M1": round(roc_auc_score(y[msk], p1[msk]), 4),
                 "gain": round(roc_auc_score(y[msk], p1[msk]) - roc_auc_score(y[msk], p0[msk]), 4)})
grad = pd.DataFrame(grad); grad.to_csv(f"{TAB}/study1_horizon_gradient.csv", index=False)
print("\nPseudo-time gradient:\n", grad.to_string(index=False))

# ---- Fig 4: ROC curves M0 vs M1 ----
fig, ax = plt.subplots(figsize=(4.0, 4.0))
f0, t0, _ = roc_curve(y, p0); f1, t1, _ = roc_curve(y, p1)
ax.plot(f0, t0, color="0.55", linestyle="--", linewidth=1.3, label=f"M0 base grade (AUC {auc0:.3f})")
ax.plot(f1, t1, color="0.1", linestyle="-", linewidth=1.3, label=f"M1 + 2-proxy (AUC {auc1:.3f})")
ax.plot([0, 1], [0, 1], color="0.8", linewidth=0.8)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(loc="lower right", fontsize=8, frameon=False)
save_fig(fig, "fig04_study1_roc_m0_m1")

# ---- Fig 5: AUC by horizon (M0 vs M1) ----
fig, ax = plt.subplots(figsize=(5.0, 3.0)); x = np.arange(5); w = 0.38
ax.bar(x - w/2, grad["AUC_M0"], w, color="0.65", edgecolor="black", linewidth=0.6, label="M0 base grade")
ax.bar(x + w/2, grad["AUC_M1"], w, color="0.25", edgecolor="black", linewidth=0.6, label="M1 + 2-proxy")
ax.set_xticks(x); ax.set_xticklabels([f"{h}year" for h in range(1, 6)]); ax.set_ylim(0.6, 0.85)
ax.set_xlabel("Forecast horizon file"); ax.set_ylabel("Out-of-fold AUC"); ax.legend(fontsize=8, frameon=False)
save_fig(fig, "fig05_study1_auc_by_horizon")

# ---- Fig 6: H4 incremental AUC (M1-M0) within mid band vs extremes ----
mid_mask = D["mid"].values == 1
gain_mid = roc_auc_score(y[mid_mask], p1[mid_mask]) - roc_auc_score(y[mid_mask], p0[mid_mask])
gain_ext = roc_auc_score(y[~mid_mask], p1[~mid_mask]) - roc_auc_score(y[~mid_mask], p0[~mid_mask])
fig, ax = plt.subplots(figsize=(3.6, 3.0))
ax.bar(["Mid band\n(B/BB analog)", "Extremes"], [gain_mid, gain_ext], color=["0.25", "0.7"], edgecolor="black", linewidth=0.7)
ax.set_ylabel("Incremental AUC (M1 - M0)")
save_fig(fig, "fig06_study1_h4_band_gain")

print("\n[02_study1] done. Figures + tables written to ../results/.")
